In [ ]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, r2_score



# =====================================================================

# 1. CÀRREGA DEL DATASET REAL (Les teves 1068 files)

# =====================================================================

df = pd.read_csv('./data/processed/03_master_dataset_final.csv')



# Seleccionem NOMÉS les 3 variables estrella del TFG

columnes_x = [

    'renda_disponible_Import_Euros', 

    'edificacions_any_Any_Mitja_Ponderat',

    'pad_dom_Llars_1_Avi_Sol'

]



# Netegem possibles NaNs que hagin quedat a aquestes 3 columnes

df_clean = df.dropna(subset=columnes_x).copy()

X = df_clean[columnes_x]



# =====================================================================

# 2. CREACIÓ DE LA VARIABLE RESPOSTA (Y) SIMULADA

# =====================================================================

np.random.seed(42)



# Lògica: Renda baixa + Pis vell + Avis = Risc Alt

y_simulada = (

    (60000 - X['renda_disponible_Import_Euros']) * 0.001 + 

    (2025 - X['edificacions_any_Any_Mitja_Ponderat']) * 0.2 + 

    (X['pad_dom_Llars_1_Avi_Sol'] * 0.15)

)



# Limitem l'Índex entre 0 i 100 i hi posem soroll

y_simulada = np.clip(y_simulada + np.random.normal(0, 4, len(X)), 0, 100)



# =====================================================================

# 3. PIPELINE DE MACHINE LEARNING

# =====================================================================

X_train, X_test, y_train, y_test = train_test_split(X, y_simulada, test_size=0.15, random_state=42)



scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)



# Executem els 3 models

models = {

    "Regressió Lineal": LinearRegression(),

    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),

    "Xarxa Neuronal (MLP)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42)

}



print(f"🚀 Llançant prova amb {len(df_clean)} seccions reals de Barcelona...\n")



for nom, model in models.items():

    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)

    print(f"--- {nom} ---")

    print(f"MAE: {mean_absolute_error(y_test, preds):.2f} | R2: {r2_score(y_test, preds):.3f}\n")



# Importància de variables (Només del Random Forest)

rf_model = models["Random Forest"]

print("🔍 IMPORTÀNCIA DE LES VARIABLES (Només disponible al Random Forest):")

for nom_var, imp in zip(columnes_x, rf_model.feature_importances_):

    print(f" - {nom_var}: {imp*100:.1f}%")